## Example 1: OAuth1 session setup

In [ ]:
# Example 1: OAuth1 session setup
from requests_oauthlib import OAuth1Session
import requests_oauthlib

print("Testing requests-oauthlib OAuth1:")

# OAuth1 credentials
client_key = 'demo_client_key'
client_secret = 'demo_secret'

# Create OAuth1 session
oauth1_session = OAuth1Session(client_key, client_secret=client_secret)

print(f"✓ OAuth1Session created: {type(oauth1_session).__name__}")
print(f"✓ Client key: {oauth1_session.auth.client.client_key}")
print(f"✓ Signature method: {oauth1_session.auth.client.signature_method}")
print(f"✓ Signature type: {oauth1_session.auth.client.signature_type}")

# Check session capabilities
print(f"✓ Session auth: {type(oauth1_session.auth).__name__}")
print(f"✓ Package version: {requests_oauthlib.__version__}")

assert oauth1_session.auth.client.client_key == 'demo_client_key'
assert oauth1_session.auth.client.signature_method == 'HMAC-SHA1'
assert oauth1_session.auth.client.signature_type == 'AUTH_HEADER'

# The session's auth object signs a prepared request offline: it adds an OAuth Authorization header
import requests
prepared = requests.Request('GET', 'https://api.example.com/resource?x=1').prepare()
header = oauth1_session.auth(prepared).headers['Authorization']
header = header.decode() if isinstance(header, bytes) else header
print(f"✓ Authorization header: {header[:60]}...")
assert header.startswith('OAuth '), header
for field in ('oauth_nonce=', 'oauth_timestamp=', 'oauth_signature=', 'oauth_signature_method="HMAC-SHA1"',
              'oauth_consumer_key="demo_client_key"'):
    assert field in header, (field, header)

## Example 2: OAuth2 session and authorization

In [ ]:
# Example 2: OAuth2 session and authorization
from requests_oauthlib import OAuth2Session

print("Testing requests-oauthlib OAuth2:")

# OAuth2 credentials
client_id = 'demo_client_id'
redirect_uri = 'https://example.com/callback'
authorization_base_url = 'https://provider.com/oauth/authorize'

# Create OAuth2 session
oauth2_session = OAuth2Session(client_id, redirect_uri=redirect_uri)

print(f"✓ OAuth2Session created: {type(oauth2_session).__name__}")
print(f"✓ Client ID: {oauth2_session.client_id}")
print(f"✓ Redirect URI: {oauth2_session.redirect_uri}")

# Generate authorization URL
authorization_url, state = oauth2_session.authorization_url(authorization_base_url)

print(f"✓ Authorization URL: {len(authorization_url)} chars")
print(f"✓ State: {state[:10]}...")
print(f"✓ Contains client_id: {'client_id' in authorization_url}")
print(f"✓ Contains redirect_uri: {'redirect_uri' in authorization_url}")

from urllib.parse import urlparse, parse_qs
assert oauth2_session.client_id == client_id and oauth2_session.redirect_uri == redirect_uri
assert authorization_url.startswith(authorization_base_url + '?'), authorization_url
query = parse_qs(urlparse(authorization_url).query)
assert query == {'response_type': ['code'], 'client_id': [client_id], 'redirect_uri': [redirect_uri], 'state': [state]}, query
assert len(state) == 30, state  # the default state is a 30-character random token

## Example 3: OAuth1 with custom configuration

In [ ]:
# Example 3: OAuth1 with custom configuration
from requests_oauthlib import OAuth1Session
from oauthlib.oauth1 import SIGNATURE_HMAC, SIGNATURE_TYPE_AUTH_HEADER

print("Testing OAuth1 custom configuration:")

# OAuth1 with custom settings
client_key = 'demo_key'
client_secret = 'demo_secret'
resource_owner_key = 'demo_token'
resource_owner_secret = 'ro_secret'

# Create OAuth1 session with custom settings
oauth1_custom = OAuth1Session(
    client_key,
    client_secret=client_secret,
    resource_owner_key=resource_owner_key,
    resource_owner_secret=resource_owner_secret,
    signature_method=SIGNATURE_HMAC,
    signature_type=SIGNATURE_TYPE_AUTH_HEADER
)

print(f"✓ Custom OAuth1 session: {type(oauth1_custom).__name__}")
print(f"✓ Signature method: {oauth1_custom.auth.client.signature_method}")
print(f"✓ Signature type: {oauth1_custom.auth.client.signature_type}")
print(f"✓ Resource owner key: {oauth1_custom.auth.client.resource_owner_key}")

assert oauth1_custom.auth.client.signature_method == SIGNATURE_HMAC
assert oauth1_custom.auth.client.signature_type == SIGNATURE_TYPE_AUTH_HEADER
assert oauth1_custom.auth.client.resource_owner_key == 'demo_token'

# With a resource owner key the signed header also carries oauth_token
import requests
prepared = requests.Request('POST', 'https://api.example.com/r', data={'a': '1'}).prepare()
header = oauth1_custom.auth(prepared).headers['Authorization']
header = header.decode() if isinstance(header, bytes) else header
assert 'oauth_token="demo_token"' in header and 'oauth_consumer_key="demo_key"' in header, header

## Example 4: OAuth2 backend application flow

In [ ]:
# Example 4: OAuth2 Backend Application flow
from requests_oauthlib import OAuth2Session
from oauthlib.oauth2 import BackendApplicationClient

print("Testing OAuth2 Backend Application flow:")

# OAuth2 Client Credentials setup
client_id = 'backend_client_id'
client_secret = 'demo_secret'
token_url = 'https://provider.com/oauth/token'

# Create backend application client
client = BackendApplicationClient(client_id=client_id)
oauth2_backend = OAuth2Session(client=client)

print(f"✓ Backend OAuth2 session: {type(oauth2_backend).__name__}")
print(f"✓ Client ID: {oauth2_backend.client_id}")
print(f"✓ Client type: {type(client).__name__}")

# Token request data
token_request_data = {
    'grant_type': 'client_credentials',
    'scope': 'read write'
}

print(f"✓ Grant type: {token_request_data['grant_type']}")
print(f"✓ Scopes: {token_request_data['scope']}")

assert oauth2_backend.client_id == 'backend_client_id'
assert isinstance(client, BackendApplicationClient)
# The client-credentials token request body that fetch_token would send
body = client.prepare_request_body(scope=token_request_data['scope'].split())
print(f"✓ Token request body: {body}")
assert body == 'grant_type=client_credentials&scope=read+write', body
assert not oauth2_backend.authorized  # no token has been fetched

## Example 5: OAuth1/OAuth2 capabilities

In [ ]:
# Example 5: Requests-OAuthlib capabilities
from requests_oauthlib import OAuth1Session, OAuth2Session
import requests_oauthlib
from oauthlib.oauth2 import WebApplicationClient, BackendApplicationClient

print("Testing requests-oauthlib capabilities:")

# OAuth1 capabilities
oauth1_demo = OAuth1Session('demo_key', client_secret='demo_secret')
print(f"✓ OAuth1Session: {type(oauth1_demo).__name__}")
print(f"✓ OAuth1 auth type: {type(oauth1_demo.auth).__name__}")

# OAuth2 capabilities  
web_client = WebApplicationClient('demo_id')
oauth2_web = OAuth2Session('demo_id', client=web_client)

backend_client = BackendApplicationClient('demo_id')
oauth2_backend = OAuth2Session('demo_id', client=backend_client)

print(f"✓ OAuth2Session: {type(oauth2_web).__name__}")
print(f"✓ WebApplicationClient: {type(web_client).__name__}")
print(f"✓ BackendApplicationClient: {type(backend_client).__name__}")

# Package info
print(f"✓ Package version: {requests_oauthlib.__version__}")
print(f"✓ OAuth1 methods: {len([m for m in dir(oauth1_demo) if not m.startswith('_')])}")
print(f"✓ OAuth2 methods: {len([m for m in dir(oauth2_web) if not m.startswith('_')])}")

assert type(oauth1_demo.auth).__name__ == 'OAuth1'
assert oauth2_web.client_id == 'demo_id' and oauth2_backend.client_id == 'demo_id'
assert isinstance(requests_oauthlib.__version__, str) and requests_oauthlib.__version__